# LLM Baseline

A simple starting point for querying a local LLM via [Ollama](https://ollama.com).

### Setup
1. Install Ollama: https://ollama.com/download
2. Pull a model: `ollama pull mistral`
3. Make sure Ollama is running: `ollama serve`
4. Run this notebook

---

## 1. Configuration

Change `MODEL` to whichever model you pulled.  
Run `ollama list` in a terminal to see what you have.

In [9]:
MODEL = "mistral"   # change to: llama3, gemma3, phi4, etc.
OLLAMA_URL = "http://localhost:11434/api/generate"

In [13]:
import sys
!{sys.executable} -m pip install requests

  Using cached requests-2.33.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Ask the LLM

In [14]:
import requests

def ask(prompt: str, system: str = "") -> str:
    """Send a prompt to the local LLM and return its response."""
    full_prompt = f"{system}\n\n{prompt}" if system else prompt

    response = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": full_prompt,
        "stream": False,
    })
    response.raise_for_status()
    return response.json()["response"]


# --- Try it ---
answer = ask("What is 2 + 2? Be brief.")
print(answer)

 The answer is 4.


## 3. Use a System Prompt

A system prompt sets the behaviour/persona of the model. Customize this for your task.

In [15]:
SYSTEM_PROMPT = """You are a helpful assistant. Answer clearly and concisely."""

# TODO: replace this with your own query
my_query = "Explain what a neural network is in two sentences."

result = ask(my_query, system=SYSTEM_PROMPT)
print(result)

 A neural network is a series of algorithms modeled after the structure and function of neurons in a brain, designed to recognize patterns in data through learning and iterative improvement. It's used for tasks such as image recognition, speech recognition, machine translation, and natural language processing.


## 4. Run Over a List of Inputs

Replace `my_queries` with your own dataset or CSV.

In [22]:
import sys
!{sys.executable} -m pip install pandas

  Using cached pandas-3.0.1-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp312-cp312-win_amd64.whl (9.7 MB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --------------- ------------------------ 4.7/12.3 MB 25.9 MB/s eta 0:00:01
   -------------------------------- ------- 10.0/12.3 MB 24.9 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 22.7 MB/s eta 0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from tqdm.notebook import tqdm

# TODO: replace with your actual queries (or load from a CSV)
my_queries = [
    "What is machine learning?",
    "What is the difference between supervised and unsupervised learning?",
    "Give one real-world use case of NLP.",
]

results = []
for query in (my_queries):
    response = ask(query, system=SYSTEM_PROMPT)
    results.append({"query": query, "response": response})

# Preview
for r in results:
    print(f"Q: {r['query']}")
    print(f"A: {r['response']}")
    print("-" * 60)

Q: What is machine learning?
A:  Machine learning is a subset of artificial intelligence that involves training algorithms to learn patterns in data, allowing them to make decisions or predictions based on new information without being explicitly programmed for each specific case. It's used in various applications, including image recognition, natural language processing, and predictive analytics.
------------------------------------------------------------
Q: What is the difference between supervised and unsupervised learning?
A:  In supervised learning, the algorithm learns from labeled data, meaning that the input data comes with predefined outputs or answers. The goal of the algorithm is to learn a mapping function between the inputs and outputs so it can accurately predict the output for new inputs. Examples include classification and regression tasks.

On the other hand, unsupervised learning involves dealing with unlabeled data where no correct answers are provided. Instead, the

## 5. Save Results

In [23]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("output.csv", index=False)

print("Saved to output.csv")
df.head()

Saved to output.csv


,query,response
0,What is machine learning?,Machine learning is a subset of artificial in...
1,What is the difference between supervised and ...,"In supervised learning, the algorithm learns ..."
2,Give one real-world use case of NLP.,One real-world use case of Natural Language P...


---
## What to customize

| What | Where |
|---|---|
| Swap the model | Cell 1 — change `MODEL` |
| Change the task | Cell 3 — rewrite `SYSTEM_PROMPT` |
| Add your data | Cell 4 — replace `my_queries` with a CSV load |
| Post-process output | After the `ask()` call — parse, extract, score, etc. |